# 01 — Build Production Feature Panel

## Purpose

Build the clean production feature panel that will feed the recalibration/backtest notebooks.

This notebook **does not**:
- run parameter sweeps
- select trades
- classify Core / Secondary
- size trades
- include risk management
- optimize P&L or tail risk

It only creates the time series needed for the simplified production system.

## Production feature definition

For each signal timestamp/date and tenor:

```text
VRP = log(implied_variance / forecast_variance)
```

Feature columns produced:

```text
date / timestamp
tenor
tenor_group
spx_close or signal_price
implied_variance
forecast_variance
vrp_log
vrp_z_3m
vrp_z_1y
rv21d
rsi14
```

## RSI timing convention

Use the **current signal-time RSI**.

For daily backtests, that means the RSI computed using that day's SPX close.

For future intraday runs, that means the RSI computed using the latest available SPX price at the time the signal is run, not the prior business day's RSI.


In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Project paths

Set `PROJECT_ROOT` to the folder that contains your `data` folder.

If this notebook lives in something like:

```text
C:\Users\patri\vrp_project\notebooks v1
```

then `PROJECT_ROOT` should be:

```text
C:\Users\patri\vrp_project
```


In [2]:
# Adjust manually if needed.
# Example:
# PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    # Fallback for Patrick's local project structure. Edit if your folder is different.
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"
OUTPUT_DIR = PROCESSED_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Data dir exists:", DATA_DIR.exists())
print("Processed dir:", PROCESSED_DIR)


Current working directory: C:\Users\patri\vrp_project\notebooks v1
Project root: C:\Users\patri\vrp_project
Data dir exists: True
Processed dir: C:\Users\patri\vrp_project\data\processed


## 2. Input source

This notebook can build the feature panel from either:

### Preferred source: one combined panel

A combined panel should contain at least:

```text
date / trade_date / timestamp
tenor / target_tenor / tenor_dte
implied_variance
forecast_variance or forecast_vol
spx_close
```

### Fallback source: separate panels

Separate inputs can be merged if needed:

```text
SPX prices
implied variance panel
forecast variance panel
```

For the restart, the simplest path is usually to use an existing processed VRP/forecast panel as the input, then rebuild clean features from it.


In [3]:
# Optional manual overrides. Leave as None for auto-detection.
COMBINED_PANEL_PATH = None

SPX_PRICES_PATH = None
IMPLIED_VARIANCE_PATH = None
FORECAST_VARIANCE_PATH = None

# Candidate combined panels to try first, in order.
COMBINED_PANEL_CANDIDATES = [
    PROCESSED_DIR / "forecast_vrp_zscore_panel_v0_1.parquet",
    PROCESSED_DIR / "forecast_vrp_panel_v0_1.parquet",
    PROCESSED_DIR / "forecast_trade_signal_panel_v0_1.parquet",
    PROCESSED_DIR / "forecast_vrp_champion_zscore_panel_v0_1.parquet",
    PROCESSED_DIR / "vrp_zscore_panel_v0_1.parquet",
    PROCESSED_DIR / "vrp_panel_v0_1.parquet",
]

def existing(paths):
    return [Path(p) for p in paths if p is not None and Path(p).exists()]

print("Existing combined-panel candidates:")
for p in existing(COMBINED_PANEL_CANDIDATES):
    print(" -", p)


Existing combined-panel candidates:
 - C:\Users\patri\vrp_project\data\processed\forecast_vrp_zscore_panel_v0_1.parquet
 - C:\Users\patri\vrp_project\data\processed\forecast_vrp_panel_v0_1.parquet
 - C:\Users\patri\vrp_project\data\processed\forecast_trade_signal_panel_v0_1.parquet
 - C:\Users\patri\vrp_project\data\processed\forecast_vrp_champion_zscore_panel_v0_1.parquet
 - C:\Users\patri\vrp_project\data\processed\vrp_zscore_panel_v0_1.parquet
 - C:\Users\patri\vrp_project\data\processed\vrp_panel_v0_1.parquet


In [4]:
def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    print(f"Reading: {path}")

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Unsupported file type: {path}")


def find_first_existing(candidates, label: str) -> Path:
    candidates = [Path(p) for p in candidates if p is not None]
    found = [p for p in candidates if p.exists()]

    if found:
        print(f"{label}: using {found[0]}")
        return found[0]

    checked = "\n".join(f"  - {p}" for p in candidates)
    raise FileNotFoundError(f"No {label} found. Checked:\n{checked}")


def choose_col(df: pd.DataFrame, candidates, required=True, label=None):
    cols_lower = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        if c in df.columns:
            return c

        c_lower = str(c).lower()
        if c_lower in cols_lower:
            return cols_lower[c_lower]

    if required:
        raise KeyError(
            f"Could not find column for {label or candidates}. "
            f"Tried: {candidates}. Available: {list(df.columns)}"
        )

    return None


def parse_project_date_series(s: pd.Series) -> pd.Series:
    """
    Robust parser for this project.

    Handles:
      - integer YYYYMMDD, e.g. 20180625
      - string YYYYMMDD, e.g. '20180625'
      - datetime strings
      - existing datetime columns

    This avoids pandas interpreting integer dates as nanoseconds after 1970-01-01.
    """
    s = s.copy()

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s)

    # Convert to string while preserving missing values.
    s_str = s.astype("string").str.strip()

    # Convert 20180625.0 -> 20180625 if needed.
    s_str = s_str.str.replace(r"\.0$", "", regex=True)

    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    # Case 1: YYYYMMDD
    is_yyyymmdd = s_str.str.fullmatch(r"\d{8}", na=False)

    if is_yyyymmdd.any():
        parsed.loc[is_yyyymmdd] = pd.to_datetime(
            s_str.loc[is_yyyymmdd],
            format="%Y%m%d",
            errors="coerce",
        )

    # Case 2: normal datetime-looking strings
    remaining = ~is_yyyymmdd

    if remaining.any():
        parsed.loc[remaining] = pd.to_datetime(
            s_str.loc[remaining],
            errors="coerce",
        )

    return parsed


def standardize_date_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    date_col = choose_col(
        df,
        ["trade_date", "date", "timestamp", "datetime", "trade_date_ts"],
        label="date/timestamp",
    )

    print("Date column used:", date_col)
    print("Raw date sample:")
    print(df[date_col].head(10).to_string(index=False))

    df["timestamp"] = parse_project_date_series(df[date_col])
    df["date"] = df["timestamp"].dt.normalize()

    bad_dates = int(df["date"].isna().sum())

    if bad_dates > 0:
        print(f"WARNING: {bad_dates} rows have unparseable dates.")
        display(df.loc[df["date"].isna(), [date_col]].head(20))

    min_date = df["date"].min()
    max_date = df["date"].max()

    print("Parsed date range:", min_date, "to", max_date)

    if pd.notna(max_date) and max_date.year < 2000:
        raise ValueError(
            "Date parsing failed: parsed dates are before 2000. "
            "This usually means integer YYYYMMDD dates were interpreted incorrectly."
        )

    return df


def standardize_tenor_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    tenor_col = choose_col(
        df,
        ["tenor", "target_tenor", "tenor_dte", "dte", "actual_dte"],
        label="tenor",
    )

    print("Tenor column used:", tenor_col)

    df["tenor"] = pd.to_numeric(df[tenor_col], errors="coerce").astype("Int64")

    return df


def tenor_group_from_tenor(tenor):
    if pd.isna(tenor):
        return np.nan

    tenor = int(tenor)

    if 9 <= tenor <= 15:
        return "front"

    if 18 <= tenor <= 24:
        return "middle"

    if 27 <= tenor <= 33:
        return "back"

    return "other"


def clean_positive_numeric(s):
    x = pd.to_numeric(s, errors="coerce")
    return x.where(x > 0)

## 3. Load input data

The notebook first tries to load a combined panel. If that fails, it can be extended to merge separate panels.

For the first production rebuild, using a combined historical panel is the most controlled path.


In [5]:
if COMBINED_PANEL_PATH is not None:
    combined_path = Path(COMBINED_PANEL_PATH)
else:
    combined_path = find_first_existing(COMBINED_PANEL_CANDIDATES, "combined panel")

raw = read_table(combined_path)

print("Raw shape:", raw.shape)
print("Raw columns:")
print(list(raw.columns))
raw.head()


combined panel: using C:\Users\patri\vrp_project\data\processed\forecast_vrp_zscore_panel_v0_1.parquet
Reading: C:\Users\patri\vrp_project\data\processed\forecast_vrp_zscore_panel_v0_1.parquet
Raw shape: (18099, 34)
Raw columns:
['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'trailing_realized_variance', 'trailing_realized_vol', 'primary_vrp_signal', 'scheduled_forward_num_returns', 'ewma_10_scheduled_forecast_variance', 'ewma_10_scheduled_forecast_vol', 'ewma_10_scheduled_vrp_signal', 'ewma_10_bias_adjustment_log', 'ewma_10_scheduled_bias_adjusted_forecast_variance', 'ewma_10_scheduled_bias_adjusted_forecast_vol', 'ewma_10_scheduled_bias_adjusted_vrp_signal', 'simple_forecast_model', 'simple_forecast_variance', 'simple_forecast_vol', 'simple_forecast_vrp_signal', 'simple_forecast_vrp_vol_ratio', 'forward_realized_variance', 'forward_realized_vol', 'forecast_vrp_3m_z_rolling_mean', 'forecast_vrp_3m_z_rolling_std', 'forecast_vrp_3m_z', 'forecast_

,trade_date,tenor,spx_close,spx_rsi_14,vix_style_vol,implied_variance,trailing_realized_variance,trailing_realized_vol,primary_vrp_signal,scheduled_forward_num_returns,ewma_10_scheduled_forecast_variance,ewma_10_scheduled_forecast_vol,ewma_10_scheduled_vrp_signal,ewma_10_bias_adjustment_log,ewma_10_scheduled_bias_adjusted_forecast_variance,ewma_10_scheduled_bias_adjusted_forecast_vol,ewma_10_scheduled_bias_adjusted_vrp_signal,simple_forecast_model,simple_forecast_variance,simple_forecast_vol,simple_forecast_vrp_signal,simple_forecast_vrp_vol_ratio,forward_realized_variance,forward_realized_vol,forecast_vrp_3m_z_rolling_mean,forecast_vrp_3m_z_rolling_std,forecast_vrp_3m_z,forecast_vrp_1y_z_rolling_mean,forecast_vrp_1y_z_rolling_std,forecast_vrp_1y_z,forecast_vrp_blended_z,forecast_rank_3m_z_highest_richest,forecast_rank_1y_z_highest_richest,forecast_rank_blended_z_highest_richest
0,20180625,9,2717.07,41.761584,17.348587,0.030097,0.010491,10.242504,1.053930,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,0.006165,7.851987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20180626,9,2723.06,43.759005,15.359305,0.023591,0.010688,10.338066,0.791777,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,0.008957,9.464090,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20180627,9,2699.63,38.235030,17.866806,0.031922,0.013532,11.632867,0.858220,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,0.008821,9.392063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20180628,9,2716.31,43.685260,15.956605,0.025461,0.014412,12.004960,0.569106,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,0.007282,8.533681,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20180629,9,2718.37,44.338515,16.193530,0.026223,0.014317,11.965170,0.605224,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,0.007259,8.520014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Standardize core columns

This section maps the source panel into production column names.

The notebook prefers direct variance columns. If only vol-point columns exist, it converts vol to variance as:

```text
variance = (vol_points / 100)^2
```


In [6]:
df = raw.copy()
df = standardize_date_column(df)
df = standardize_tenor_column(df)

df["tenor_group"] = df["tenor"].apply(tenor_group_from_tenor)

# SPX close / signal price
spx_col = choose_col(df, ["spx_close", "close", "spot", "underlying_price", "entry_spx_close"], label="SPX close")
df["spx_close"] = pd.to_numeric(df[spx_col], errors="coerce")

# Implied variance
implied_var_col = choose_col(df, ["implied_variance", "implied_var", "ivariance", "iv_var"], required=False)

if implied_var_col is not None:
    df["implied_variance"] = clean_positive_numeric(df[implied_var_col])
else:
    implied_vol_col = choose_col(df, ["vix_style_vol", "implied_vol", "implied_volatility", "iv"], label="implied vol")
    df["implied_variance"] = (pd.to_numeric(df[implied_vol_col], errors="coerce") / 100.0) ** 2

# Forecast variance.
# Prefer an explicitly named forecast variance column.
forecast_var_col = choose_col(
    df,
    [
        "forecast_variance",
        "forecast_var",
        "predicted_variance",
        "model_variance",
        "har_rv_iv_forecast_variance",
        "simple_forecast_variance",
    ],
    required=False,
)

if forecast_var_col is not None:
    df["forecast_variance"] = clean_positive_numeric(df[forecast_var_col])
else:
    forecast_vol_col = choose_col(
        df,
        [
            "forecast_vol",
            "forecast_volatility",
            "predicted_vol",
            "model_vol",
            "har_rv_iv_forecast_vol",
        ],
        label="forecast vol",
    )
    df["forecast_variance"] = (pd.to_numeric(df[forecast_vol_col], errors="coerce") / 100.0) ** 2

# Keep production tenor grid only.
PRODUCTION_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
df = df[df["tenor"].isin(PRODUCTION_TENORS)].copy()

print("Standardized shape:", df.shape)
print(df[["date", "tenor", "tenor_group", "spx_close", "implied_variance", "forecast_variance"]].head(10))


Date column used: trade_date
Raw date sample:
20180625
20180626
20180627
20180628
20180629
20180702
20180703
20180705
20180706
20180709
Parsed date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
Tenor column used: tenor
Standardized shape: (18099, 38)
        date  tenor tenor_group  spx_close  implied_variance  forecast_variance
0 2018-06-25      9       front    2717.07          0.030097                NaN
1 2018-06-26      9       front    2723.06          0.023591                NaN
2 2018-06-27      9       front    2699.63          0.031922                NaN
3 2018-06-28      9       front    2716.31          0.025461                NaN
4 2018-06-29      9       front    2718.37          0.026223                NaN
5 2018-07-02      9       front    2726.71          0.020917                NaN
6 2018-07-03      9       front    2713.22          0.023379                NaN
7 2018-07-05      9       front    2736.61          0.020276                NaN
8 2018-07-06      9      

## 5. Compute current signal-time RSI and 21d realized vol

This recomputes `rsi14` and `rv21d` from the signal-time SPX price series.

For daily backtests, `spx_close` is the current day's close.

For future intraday runs, the same logic should be applied to a series where the final observation is the current intraday SPX price.


In [7]:
def compute_wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    close = pd.to_numeric(close, errors="coerce")
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)

    # Wilder smoothing is equivalent to EWM with alpha = 1 / period.
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def compute_realized_vol_21d(close: pd.Series, window: int = 21, annualization: int = 252) -> pd.Series:
    close = pd.to_numeric(close, errors="coerce")
    log_ret = np.log(close).diff()
    return log_ret.rolling(window=window, min_periods=window).std(ddof=1) * np.sqrt(annualization) * 100

# One SPX price per date/timestamp.
price_series = (
    df[["date", "spx_close"]]
    .dropna()
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .set_index("date")["spx_close"]
)

market_features = pd.DataFrame(index=price_series.index)
market_features["spx_close_for_indicators"] = price_series
market_features["rsi14"] = compute_wilder_rsi(price_series, period=14)
market_features["rv21d"] = compute_realized_vol_21d(price_series, window=21, annualization=252)
market_features = market_features.reset_index()

df = df.merge(market_features[["date", "rsi14", "rv21d"]], on="date", how="left")

print(df[["date", "tenor", "spx_close", "rsi14", "rv21d"]].head(20))


         date  tenor  spx_close      rsi14  rv21d
0  2018-06-25      9    2717.07        NaN    NaN
1  2018-06-26      9    2723.06        NaN    NaN
2  2018-06-27      9    2699.63        NaN    NaN
3  2018-06-28      9    2716.31        NaN    NaN
4  2018-06-29      9    2718.37        NaN    NaN
5  2018-07-02      9    2726.71        NaN    NaN
6  2018-07-03      9    2713.22        NaN    NaN
7  2018-07-05      9    2736.61        NaN    NaN
8  2018-07-06      9    2759.82        NaN    NaN
9  2018-07-09      9    2784.17        NaN    NaN
10 2018-07-10      9    2793.84        NaN    NaN
11 2018-07-11      9    2774.02        NaN    NaN
12 2018-07-12      9    2798.29        NaN    NaN
13 2018-07-13      9    2801.31        NaN    NaN
14 2018-07-16      9    2798.43  77.177431    NaN
15 2018-07-17      9    2809.55  78.794775    NaN
16 2018-07-18      9    2815.62  79.642830    NaN
17 2018-07-19      9    2804.49  73.813633    NaN
18 2018-07-20      9    2801.83  72.448847    NaN


## 6. Compute production VRP and z-scores

`vrp_log` is:

```text
log(implied_variance / forecast_variance)
```

The z-score windows are:

```text
3-month: 63 trading days
1-year: 252 trading days
```

By default this notebook uses a **prior-window z-score** to avoid look-ahead:

```text
today's z = (today's VRP - trailing mean through prior observation) / trailing std through prior observation
```

This is the clean production convention. Change `USE_PRIOR_WINDOW_FOR_ZSCORE` only if you intentionally want same-window/in-sample Excel-style behavior.


In [8]:
USE_PRIOR_WINDOW_FOR_ZSCORE = True

Z3M_WINDOW = 63
Z1Y_WINDOW = 252

df["vrp_log"] = np.log(df["implied_variance"] / df["forecast_variance"])

def rolling_z_by_tenor(panel: pd.DataFrame, value_col: str, window: int, use_prior_window: bool = True) -> pd.Series:
    out = pd.Series(index=panel.index, dtype=float)

    for tenor, g in panel.sort_values(["tenor", "date"]).groupby("tenor", group_keys=False):
        s = g[value_col].astype(float)
        base = s.shift(1) if use_prior_window else s
        mean = base.rolling(window=window, min_periods=window).mean()
        std = base.rolling(window=window, min_periods=window).std(ddof=1)
        z = (s - mean) / std
        out.loc[g.index] = z

    return out

df["vrp_z_3m"] = rolling_z_by_tenor(df, "vrp_log", Z3M_WINDOW, USE_PRIOR_WINDOW_FOR_ZSCORE)
df["vrp_z_1y"] = rolling_z_by_tenor(df, "vrp_log", Z1Y_WINDOW, USE_PRIOR_WINDOW_FOR_ZSCORE)

print(df[["date", "tenor", "vrp_log", "vrp_z_3m", "vrp_z_1y"]].dropna(subset=["vrp_z_3m"]).head(20))


         date  tenor   vrp_log  vrp_z_3m  vrp_z_1y
72 2018-10-05      9  1.861894  2.149222       NaN
73 2018-10-08      9  1.544530  1.634026       NaN
74 2018-10-09      9  1.540741  1.591043       NaN
75 2018-10-10      9  0.770901  0.459452       NaN
76 2018-10-11      9  1.259614  1.144636       NaN
77 2018-10-12      9  0.753425  0.368850       NaN
78 2018-10-15      9  0.543965  0.024841       NaN
79 2018-10-16      9 -0.128874 -1.067843       NaN
80 2018-10-17      9 -0.109186 -1.092145       NaN
81 2018-10-18      9  0.308224 -0.425936       NaN
82 2018-10-19      9  0.575513  0.000495       NaN
83 2018-10-22      9  0.249889 -0.581588       NaN
84 2018-10-23      9  0.603161  0.002997       NaN
85 2018-10-24      9  0.771934  0.275139       NaN
86 2018-10-25      9  0.510220 -0.227819       NaN
87 2018-10-26      9  0.850157  0.385708       NaN
88 2018-10-29      9  0.690991  0.069389       NaN
89 2018-10-30      9  0.473856 -0.385816       NaN
90 2018-10-31      9  0.224626 

## 7. Quality checks

The production feature panel should have one row per date/timestamp and tenor.

It should not contain non-positive variance values.

The recalibration notebook will require both z-score windows to be available.


In [9]:
feature_cols = [
    "date",
    "tenor",
    "tenor_group",
    "spx_close",
    "implied_variance",
    "forecast_variance",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
]

# Start from the standardized dataframe produced by earlier cells.
feature_source = df.copy()

# ---------------------------------------------------------------------
# Resolve duplicate date/tenor rows.
#
# We need exactly one production feature row per date/tenor.
# Duplicates can happen because old processed panels may include multiple
# forecast models, signal variants, or research versions.
# ---------------------------------------------------------------------

key_cols = ["date", "tenor"]

dupe_count_before = feature_source.duplicated(subset=key_cols).sum()
print("Duplicate date/tenor rows before cleanup:", dupe_count_before)

if dupe_count_before > 0:
    print("\nDuplicate sample before cleanup:")
    display_cols = [
        c for c in [
            "date",
            "tenor",
            "tenor_group",
            "simple_forecast_model",
            "forecast_model",
            "model",
            "forecast_denominator",
            "denominator",
            "spx_close",
            "implied_variance",
            "forecast_variance",
            "vrp_log",
            "vrp_z_3m",
            "vrp_z_1y",
            "rv21d",
            "rsi14",
        ]
        if c in feature_source.columns
    ]

    display(
        feature_source[
            feature_source.duplicated(subset=key_cols, keep=False)
        ][display_cols]
        .sort_values(key_cols)
        .head(30)
    )

# Prefer the forecast denominator we decided to use.
# If no model-identifying column exists, all rows stay at the same model rank.
model_preference = [
    "per_tenor_har_rv_iv",
    "har_rv_iv",
    "ewma_10_scheduled_raw",
]

model_cols = [
    c for c in [
        "forecast_model",
        "simple_forecast_model",
        "model",
        "forecast_denominator",
        "denominator",
    ]
    if c in feature_source.columns
]

feature_source["_model_rank"] = 999

if model_cols:
    model_col = model_cols[0]
    print("\nModel column used for duplicate preference:", model_col)

    model_values = feature_source[model_col].astype(str)

    for rank, model_name in enumerate(model_preference):
        feature_source.loc[
            model_values.str.lower().eq(model_name.lower()),
            "_model_rank"
        ] = rank
else:
    print("\nNo model column found. Duplicate cleanup will use most-complete-row / first-row fallback.")

# Prefer rows that have all required production features populated.
feature_source["_missing_feature_count"] = feature_source[feature_cols].isna().sum(axis=1)

# Sort so preferred model / most complete row comes first.
feature_source = feature_source.sort_values(
    key_cols + ["_model_rank", "_missing_feature_count"]
)

# Check whether duplicated production features disagree.
dupe_rows = feature_source[
    feature_source.duplicated(subset=key_cols, keep=False)
].copy()

if len(dupe_rows) > 0:
    # Only check non-key feature columns for disagreement.
    # Do NOT include date/tenor in the aggregated columns, because they are already groupby keys.
    non_key_feature_cols = [c for c in feature_cols if c not in key_cols]

    disagreement = (
        dupe_rows
        .groupby(key_cols)[non_key_feature_cols]
        .nunique(dropna=False)
        .reset_index()
    )

    varying_feature_cols = [
        c for c in non_key_feature_cols
        if disagreement[c].max() > 1
    ]

    if varying_feature_cols:
        print("\nWARNING: Duplicate date/tenor rows have different production feature values.")
        print("Varying columns:", varying_feature_cols)
        print("Keeping the preferred/most complete row based on model rank and missing-feature count.")
    else:
        print("\nDuplicate rows have identical production features. Collapsing duplicates.")

# Keep one row per date/tenor.
feature_panel = (
    feature_source
    .drop_duplicates(subset=key_cols, keep="first")
    [feature_cols]
    .copy()
    .sort_values(["date", "tenor"])
    .reset_index(drop=True)
)

# Final duplicate check.
dupes = feature_panel.duplicated(subset=key_cols).sum()
print("\nDuplicate date/tenor rows after cleanup:", dupes)

if dupes > 0:
    display(
        feature_panel[
            feature_panel.duplicated(subset=key_cols, keep=False)
        ].head(20)
    )
    raise ValueError("Feature panel still has duplicate date/tenor rows after cleanup.")

# Basic quality summary.
summary = {
    "rows": len(feature_panel),
    "start_date": str(feature_panel["date"].min().date()) if len(feature_panel) else None,
    "end_date": str(feature_panel["date"].max().date()) if len(feature_panel) else None,
    "tenors": sorted(feature_panel["tenor"].dropna().unique().tolist()),
    "missing_implied_variance": int(feature_panel["implied_variance"].isna().sum()),
    "missing_forecast_variance": int(feature_panel["forecast_variance"].isna().sum()),
    "missing_vrp_log": int(feature_panel["vrp_log"].isna().sum()),
    "missing_vrp_z_3m": int(feature_panel["vrp_z_3m"].isna().sum()),
    "missing_vrp_z_1y": int(feature_panel["vrp_z_1y"].isna().sum()),
    "missing_rv21d": int(feature_panel["rv21d"].isna().sum()),
    "missing_rsi14": int(feature_panel["rsi14"].isna().sum()),
}

import json

print("\nFeature panel summary:")
print(json.dumps(summary, indent=2))

display(feature_panel.head(20))

Duplicate date/tenor rows before cleanup: 0

Model column used for duplicate preference: simple_forecast_model

Duplicate date/tenor rows after cleanup: 0

Feature panel summary:
{
  "rows": 18099,
  "start_date": "2018-06-25",
  "end_date": "2026-06-25",
  "tenors": [
    9,
    12,
    15,
    18,
    21,
    24,
    27,
    30,
    33
  ],
  "missing_implied_variance": 0,
  "missing_forecast_variance": 81,
  "missing_vrp_log": 81,
  "missing_vrp_z_3m": 648,
  "missing_vrp_z_1y": 2349,
  "missing_rv21d": 189,
  "missing_rsi14": 126
}


,date,tenor,tenor_group,spx_close,implied_variance,forecast_variance,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14
0,2018-06-25,9,front,2717.07,0.030097,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-06-25,12,front,2717.07,0.030592,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-06-25,15,front,2717.07,0.030593,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-06-25,18,middle,2717.07,0.030594,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-06-25,21,middle,2717.07,0.030368,NaN,NaN,NaN,NaN,NaN,NaN
5,2018-06-25,24,middle,2717.07,0.030199,NaN,NaN,NaN,NaN,NaN,NaN
6,2018-06-25,27,back,2717.07,0.030101,NaN,NaN,NaN,NaN,NaN,NaN
7,2018-06-25,30,back,2717.07,0.030033,NaN,NaN,NaN,NaN,NaN,NaN
8,2018-06-25,33,back,2717.07,0.030050,NaN,NaN,NaN,NaN,NaN,NaN
9,2018-06-26,9,front,2723.06,0.023591,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Save production feature panel

This output is the input to the recalibration notebook.

Expected next notebook:

```text
02_recalibrate_simple_signal_grid.ipynb
```


In [10]:
FEATURE_PANEL_PATH = OUTPUT_DIR / "production_feature_panel_v0_1.parquet"
FEATURE_PANEL_LATEST_PATH = OUTPUT_DIR / "production_feature_panel_latest_snapshot_v0_1.csv"
FEATURE_PANEL_AUDIT_PATH = AUDIT_DIR / "production_feature_panel_summary_v0_1.json"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

feature_panel.to_parquet(FEATURE_PANEL_PATH, index=False)

latest_date = feature_panel["date"].max()
latest_snapshot = feature_panel[feature_panel["date"] == latest_date].copy()
latest_snapshot.to_csv(FEATURE_PANEL_LATEST_PATH, index=False)

with open(FEATURE_PANEL_AUDIT_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved feature panel:", FEATURE_PANEL_PATH)
print("Saved latest snapshot:", FEATURE_PANEL_LATEST_PATH)
print("Saved audit summary:", FEATURE_PANEL_AUDIT_PATH)

display(latest_snapshot)


Saved feature panel: C:\Users\patri\vrp_project\data\processed\production_feature_panel_v0_1.parquet
Saved latest snapshot: C:\Users\patri\vrp_project\data\processed\production_feature_panel_latest_snapshot_v0_1.csv
Saved audit summary: C:\Users\patri\vrp_project\data\audit\production_feature_panel_summary_v0_1.json


,date,tenor,tenor_group,spx_close,implied_variance,forecast_variance,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14
18090,2026-06-25,9,front,7357.49,0.032229,0.014872,0.773380,0.396150,-0.183506,16.699148,46.297996
18091,2026-06-25,12,front,7357.49,0.032273,0.016286,0.683950,0.109394,-0.503778,16.699148,46.297996
18092,2026-06-25,15,front,7357.49,0.032300,0.019408,0.509373,-0.150917,-0.796883,16.699148,46.297996
18093,2026-06-25,18,middle,7357.49,0.031357,0.018272,0.540072,-0.181977,-0.837781,16.699148,46.297996
18094,2026-06-25,21,middle,7357.49,0.030683,0.023251,0.277368,-0.500565,-1.144971,16.699148,46.297996
18095,2026-06-25,24,middle,7357.49,0.032504,0.021798,0.399541,-0.233142,-0.921934,16.699148,46.297996
18096,2026-06-25,27,back,7357.49,0.034583,0.023251,0.397020,-0.318278,-1.027086,16.699148,46.297996
18097,2026-06-25,30,back,7357.49,0.036027,0.023251,0.437930,-0.197874,-0.935338,16.699148,46.297996
18098,2026-06-25,33,back,7357.49,0.036810,0.023251,0.459411,-0.238078,-0.985168,16.699148,46.297996


In [11]:
# Final validation cell for Notebook 1

from pathlib import Path
import pandas as pd
import json

FEATURE_PANEL_PATH = PROCESSED_DIR / "production_feature_panel_v0_1.parquet"
FEATURE_PANEL_AUDIT_PATH = AUDIT_DIR / "production_feature_panel_summary_v0_1.json"

feature_panel_check = pd.read_parquet(FEATURE_PANEL_PATH)

print("Loaded:", FEATURE_PANEL_PATH)
print("Shape:", feature_panel_check.shape)

# Basic date range
print("\nDate range:")
print("Start:", feature_panel_check["date"].min())
print("End:  ", feature_panel_check["date"].max())

# Tenor coverage
print("\nTenors:")
print(sorted(feature_panel_check["tenor"].dropna().unique().tolist()))

# Confirm one row per date / tenor
dupes = feature_panel_check.duplicated(subset=["date", "tenor"]).sum()
print("\nDuplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        feature_panel_check[
            feature_panel_check.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows found.")

# Count tenors per date
tenor_counts = (
    feature_panel_check
    .groupby("date")["tenor"]
    .nunique()
    .rename("tenor_count")
)

print("\nTenor count per date:")
print(tenor_counts.value_counts().sort_index())

bad_tenor_count_dates = tenor_counts[tenor_counts != 9]

if len(bad_tenor_count_dates) > 0:
    print("\nWARNING: Some dates do not have all 9 tenors.")
    display(bad_tenor_count_dates.head(20))
else:
    print("All dates have 9 tenors.")

# Missing feature summary
required_cols = [
    "date",
    "tenor",
    "tenor_group",
    "spx_close",
    "implied_variance",
    "forecast_variance",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
]

missing_summary = feature_panel_check[required_cols].isna().sum().sort_values(ascending=False)

print("\nMissing values:")
print(missing_summary)

# Eligible research rows: require both z windows, rv21d, and rsi14
eligible_research_rows = feature_panel_check.dropna(
    subset=[
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
    ]
)

print("\nRows eligible for recalibration after warmup:", len(eligible_research_rows))
print("Eligible start date:", eligible_research_rows["date"].min())
print("Eligible end date:  ", eligible_research_rows["date"].max())

# Latest snapshot sanity check
latest_date = feature_panel_check["date"].max()
latest_snapshot = feature_panel_check[feature_panel_check["date"] == latest_date].copy()

print("\nLatest date snapshot:", latest_date)
display(latest_snapshot.sort_values("tenor"))

# Confirm market-level filters are identical across tenors for each date
market_filter_counts = (
    feature_panel_check
    .groupby("date")[["rv21d", "rsi14"]]
    .nunique(dropna=False)
)

bad_market_filter_dates = market_filter_counts[
    (market_filter_counts["rv21d"] > 1) |
    (market_filter_counts["rsi14"] > 1)
]

print("\nDates with inconsistent rv21d/rsi14 across tenors:", len(bad_market_filter_dates))

if len(bad_market_filter_dates) > 0:
    display(bad_market_filter_dates.head(20))
    raise ValueError("rv21d or rsi14 differs across tenors on the same date.")

# Save updated validation summary
validation_summary = {
    "rows": int(len(feature_panel_check)),
    "start_date": str(feature_panel_check["date"].min().date()),
    "end_date": str(feature_panel_check["date"].max().date()),
    "tenors": sorted([int(x) for x in feature_panel_check["tenor"].dropna().unique().tolist()]),
    "duplicate_date_tenor_rows": int(dupes),
    "dates_with_all_9_tenors": int((tenor_counts == 9).sum()),
    "dates_missing_some_tenors": int((tenor_counts != 9).sum()),
    "eligible_recalibration_rows": int(len(eligible_research_rows)),
    "eligible_start_date": str(eligible_research_rows["date"].min().date()),
    "eligible_end_date": str(eligible_research_rows["date"].max().date()),
    "missing_values": {k: int(v) for k, v in missing_summary.to_dict().items()},
    "dates_with_inconsistent_market_filters": int(len(bad_market_filter_dates)),
}

with open(FEATURE_PANEL_AUDIT_PATH, "w") as f:
    json.dump(validation_summary, f, indent=2)

print("\nSaved updated validation summary:", FEATURE_PANEL_AUDIT_PATH)
print(json.dumps(validation_summary, indent=2))

Loaded: C:\Users\patri\vrp_project\data\processed\production_feature_panel_v0_1.parquet
Shape: (18099, 11)

Date range:
Start: 2018-06-25 00:00:00
End:   2026-06-25 00:00:00

Tenors:
[9, 12, 15, 18, 21, 24, 27, 30, 33]

Duplicate date/tenor rows: 0

Tenor count per date:
tenor_count
9    2011
Name: count, dtype: int64
All dates have 9 tenors.

Missing values:
vrp_z_1y             2349
vrp_z_3m              648
rv21d                 189
rsi14                 126
forecast_variance      81
vrp_log                81
date                    0
tenor_group             0
tenor                   0
spx_close               0
implied_variance        0
dtype: int64

Rows eligible for recalibration after warmup: 15750
Eligible start date: 2019-07-10 00:00:00
Eligible end date:   2026-06-25 00:00:00

Latest date snapshot: 2026-06-25 00:00:00


,date,tenor,tenor_group,spx_close,implied_variance,forecast_variance,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14
18090,2026-06-25,9,front,7357.49,0.032229,0.014872,0.773380,0.396150,-0.183506,16.699148,46.297996
18091,2026-06-25,12,front,7357.49,0.032273,0.016286,0.683950,0.109394,-0.503778,16.699148,46.297996
18092,2026-06-25,15,front,7357.49,0.032300,0.019408,0.509373,-0.150917,-0.796883,16.699148,46.297996
18093,2026-06-25,18,middle,7357.49,0.031357,0.018272,0.540072,-0.181977,-0.837781,16.699148,46.297996
18094,2026-06-25,21,middle,7357.49,0.030683,0.023251,0.277368,-0.500565,-1.144971,16.699148,46.297996
18095,2026-06-25,24,middle,7357.49,0.032504,0.021798,0.399541,-0.233142,-0.921934,16.699148,46.297996
18096,2026-06-25,27,back,7357.49,0.034583,0.023251,0.397020,-0.318278,-1.027086,16.699148,46.297996
18097,2026-06-25,30,back,7357.49,0.036027,0.023251,0.437930,-0.197874,-0.935338,16.699148,46.297996
18098,2026-06-25,33,back,7357.49,0.036810,0.023251,0.459411,-0.238078,-0.985168,16.699148,46.297996



Dates with inconsistent rv21d/rsi14 across tenors: 0

Saved updated validation summary: C:\Users\patri\vrp_project\data\audit\production_feature_panel_summary_v0_1.json
{
  "rows": 18099,
  "start_date": "2018-06-25",
  "end_date": "2026-06-25",
  "tenors": [
    9,
    12,
    15,
    18,
    21,
    24,
    27,
    30,
    33
  ],
  "duplicate_date_tenor_rows": 0,
  "dates_with_all_9_tenors": 2011,
  "dates_missing_some_tenors": 0,
  "eligible_recalibration_rows": 15750,
  "eligible_start_date": "2019-07-10",
  "eligible_end_date": "2026-06-25",
  "missing_values": {
    "vrp_z_1y": 2349,
    "vrp_z_3m": 648,
    "rv21d": 189,
    "rsi14": 126,
    "forecast_variance": 81,
    "vrp_log": 81,
    "date": 0,
    "tenor_group": 0,
    "tenor": 0,
    "spx_close": 0,
    "implied_variance": 0
  },
  "dates_with_inconsistent_market_filters": 0
}
